# Incremental Load with `COPY INTO`

**Goal:** Learn how to incrementally ingest files from cloud storage into a Delta table using `COPY INTO`, so that re-running the command only loads *new* files.

**Prerequisites**
- Unity Catalog enabled workspace (uses 3-level namespacing `catalog.schema.table`).
- A cluster or SQL warehouse on a recent DBR (this demo uses features available in **DBR 10.4 LTS and above**; `VALIDATE` requires 10.4 LTS+).
- Privileges: `USE CATALOG` / `USE SCHEMA`, `CREATE TABLE`, and `READ FILES` on the source location (a Unity Catalog Volume here).

> This notebook is **self-contained**: it generates its own sample Parquet files into a Volume, then loads them. Adjust the catalog/schema/volume names in the config cell to match your workspace.


## 0. Configuration
Set your target namespace and the source Volume path. Defaults match the `databricksansh.bronze` convention used elsewhere in this project.


In [0]:
# Python config — edit these to match your workspace
CATALOG = "databricksansh"
SCHEMA  = "bronze"
TABLE   = "orders_copyinto"
VOLUME  = "autovol"            # an existing UC managed volume in the schema
SRC_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/copyinto_demo"  # source folder for files

FQN = f"{CATALOG}.{SCHEMA}.{TABLE}"
print('Target table :', FQN)
print('Source folder:', SRC_DIR)

# Make catalog/schema the default so unqualified names resolve correctly
spark.sql(f"CREATE Catalog {CATALOG}")
spark.sql(f"Create SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')


Target table : databricksansh.bronze.orders_copyinto
Source folder: /Volumes/databricksansh/bronze/autovol/copyinto_demo


DataFrame[]

## 1. Generate sample data (batch #1)
We write two small Parquet files into the source folder to simulate files landing in cloud storage.
In real life these arrive from upstream systems — you don't create them by hand.


In [0]:
dbutils.fs.rm(SRC_DIR, recurse=True)

True

In [0]:
from pyspark.sql import Row

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
dbutils.fs.mkdirs(SRC_DIR)

batch1 = [
    Row(order_id=1, customer_id=101, order_date='2026-05-01', order_amount=120.50, order_status='SHIPPED'),
    Row(order_id=2, customer_id=102, order_date='2026-05-01', order_amount=75.00,  order_status='PENDING'),
    Row(order_id=3, customer_id=103, order_date='2026-05-02', order_amount=300.25, order_status='SHIPPED'),
]
# coalesce(1) -> one output file per write, so each batch = a distinct file
spark.createDataFrame(batch1).coalesce(1).write.mode('append').parquet(SRC_DIR)

display(dbutils.fs.ls(SRC_DIR))


path,name,size,modificationTime
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_SUCCESS,_SUCCESS,0,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_committed_3084202544796133795,_committed_3084202544796133795,125,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_started_3084202544796133795,_started_3084202544796133795,0,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,1697,1780165870000


## 2. Create the target Delta table
`COPY INTO` requires an **existing Delta table**. Define the schema explicitly (best practice for Bronze).


In [0]:
spark.sql(f'DROP TABLE IF EXISTS {FQN}')
spark.sql(f'''
CREATE TABLE {FQN} (
  order_id     INT,
  customer_id  INT,
  order_date   DATE,
  order_amount DOUBLE,
  order_status STRING
) USING DELTA
''')
print('Created', FQN)


Created databricksansh.bronze.orders_copyinto


## 3. First load
Load everything currently in the source folder. `COPY INTO` records which files it has ingested in the Delta table's transaction log.


In [0]:
spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}'
)
FILEFORMAT = PARQUET
''').display()


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
3,3,0


In [0]:
# 3 rows expected
spark.sql(f'SELECT * FROM {FQN} ORDER BY order_id').display()


order_id,customer_id,order_date,order_amount,order_status
1,101,2026-05-01,120.5,SHIPPED
2,102,2026-05-01,75.0,PENDING
3,103,2026-05-02,300.25,SHIPPED


## 4. The incremental part — re-run with new files
Now a **new file** (batch #2) lands. We re-run the *exact same* `COPY INTO` command.
Because the operation is **idempotent**, the previously loaded file is skipped and only the new file is ingested.


In [0]:
batch2 = [
    Row(order_id=4, customer_id=104, order_date='2026-05-03', order_amount=50.00,  order_status='PENDING'),
    Row(order_id=5, customer_id=101, order_date='2026-05-03', order_amount=210.75, order_status='SHIPPED'),
]
spark.createDataFrame(batch2).coalesce(1).write.mode('append').parquet(SRC_DIR)
print('Files now in source:')
display(dbutils.fs.ls(SRC_DIR))


Files now in source:


path,name,size,modificationTime
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_SUCCESS,_SUCCESS,0,1780166003000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_committed_3084202544796133795,_committed_3084202544796133795,125,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_committed_5784892464774992514,_committed_5784892464774992514,125,1780166003000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_started_3084202544796133795,_started_3084202544796133795,0,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/_started_5784892464774992514,_started_5784892464774992514,0,1780166002000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,1697,1780165870000
dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-5784892464774992514-902ca5ca-b0c6-4e62-a5f8-12c523c5b331-8430-1.c000.snappy.parquet,part-00000-tid-5784892464774992514-902ca5ca-b0c6-4e62-a5f8-12c523c5b331-8430-1.c000.snappy.parquet,1659,1780166002000


In [0]:
# Same command as before. Only the NEW file is loaded.
spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}'
)
FILEFORMAT = PARQUET
''').display()   # num_affected_rows = 2, not 5


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2,2,0


In [0]:
# 5 rows total now (3 + 2). Re-running again loads 0 rows.
spark.sql(f'SELECT * FROM {FQN} ORDER BY order_id').display()


order_id,customer_id,order_date,order_amount,order_status
1,101,2026-05-01,120.5,SHIPPED
2,102,2026-05-01,75.0,PENDING
3,103,2026-05-02,300.25,SHIPPED
4,104,2026-05-03,50.0,PENDING
5,101,2026-05-03,210.75,SHIPPED


In [0]:
# Proof of idempotency: run a third time with no new files -> 0 rows affected
spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}'
)
FILEFORMAT = PARQUET
''').display()


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


## 5. `force` — disable idempotency (reload everything)
Sometimes you need to reprocess files that were already loaded (e.g. after a bug fix).
`COPY_OPTIONS ('force' = 'true')` ignores the load history and re-ingests **all** matching files.

> ⚠️ This will create **duplicate rows** if your table already contains the data — `COPY INTO` appends, it does not de-duplicate.


In [0]:
spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}'
)
FILEFORMAT = PARQUET
COPY_OPTIONS ('force' = 'true')
''').display()   # all 5 rows re-loaded -> table now has duplicates

spark.sql(f'SELECT COUNT(*) AS row_count FROM {FQN}').display()


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5,5,0


row_count
10


In [0]:
# Reset the table for the next demo (clean slate)
spark.sql(f'TRUNCATE TABLE {FQN}')
spark.sql(f'''
          COPY INTO {FQN} FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}'
) FILEFORMAT = PARQUET
          ''').display()


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
spark.sql(f'SELECT COUNT(*) AS row_count FROM {FQN}').display()

row_count
0


## 6. Schema evolution with `mergeSchema`
A new batch arrives with an **extra column** (`order_channel`). By default `COPY INTO` would reject the mismatch.
`COPY_OPTIONS ('mergeSchema' = 'true')` evolves the **target table** schema to add the new column.

**Gotcha — two different `mergeSchema` options:**
- `COPY_OPTIONS ('mergeSchema'='true')` → evolve the **target Delta table** (add new columns).
- `FORMAT_OPTIONS ('mergeSchema'='true')` → a **Spark reader** option that reconciles schemas *across the source files* (e.g. Parquet part files that differ). They solve different problems; you sometimes need both.


In [0]:
batch3 = [
    Row(order_id=6, customer_id=105, order_date='2026-05-04', order_amount=99.99, order_status='SHIPPED', order_channel='WEB'),
]
spark.createDataFrame(batch3).coalesce(1).write.mode('append').parquet(SRC_DIR)

spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}')
FILEFORMAT = PARQUET
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'true')
''').display()

spark.sql(f'DESCRIBE {FQN}').display()   # order_channel now present


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2,2,0


col_name,data_type,comment
order_id,int,null
customer_id,int,null
order_date,date,null
order_amount,double,null
order_status,string,null


## 7. `PATTERN` and `SELECT` — filter and transform on the way in
- `PATTERN` ingests only files matching a glob (e.g. one source folder, many feeds).
- Wrapping the source in a `SELECT` lets you cast, rename, or add columns during load — including the `_metadata` file column.


In [0]:
# Example: only load parquet files, and capture the source file name + load timestamp.
# (Run against a fresh table to see the added columns.)
spark.sql(f'DROP TABLE IF EXISTS {FQN}_enriched')
spark.sql(f'''
CREATE TABLE {FQN}_enriched (
  order_id INT, customer_id INT, order_date DATE, order_amount DOUBLE,
  order_status STRING, source_file STRING, ingested_at TIMESTAMP
)''')

spark.sql(f'''
COPY INTO {FQN}_enriched
FROM (
  SELECT CAST(order_id AS INT) AS order_id,
         CAST(customer_id AS INT) AS customer_id,
         CAST(order_date AS DATE) AS order_date,
         order_amount,
         order_status,
         _metadata.file_path AS source_file,
         current_timestamp() AS ingested_at
  FROM '{SRC_DIR}'
)
FILEFORMAT = PARQUET
PATTERN = '*.parquet'
COPY_OPTIONS ('mergeSchema' = 'true')
''').display()

spark.sql(f'SELECT order_id, source_file, ingested_at FROM {FQN}_enriched ORDER BY order_id').display()


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
7,7,0


order_id,source_file,ingested_at
1,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
2,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
3,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3084202544796133795-8b4235d3-7848-4217-b014-f653665a0c9b-8420-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
4,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-5784892464774992514-902ca5ca-b0c6-4e62-a5f8-12c523c5b331-8430-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
5,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-5784892464774992514-902ca5ca-b0c6-4e62-a5f8-12c523c5b331-8430-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
6,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-1545249676714828906-2ad6eb68-32d6-4953-b95b-90ceaabd91be-8520-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z
6,dbfs:/Volumes/databricksansh/bronze/autovol/copyinto_demo/part-00000-tid-3981135343117587186-9623ec5a-2c14-4aac-8215-b0ab7f5d03dd-8519-1.c000.snappy.parquet,2026-05-30T18:38:21.280Z


## 8. `VALIDATE` — dry-run before loading
`VALIDATE` (DBR 10.4 LTS+) checks parsing, schema match, and constraints **without writing**. Returns a preview.
Use it when onboarding a new feed to catch bad data before it hits the table.


In [0]:
spark.sql(f'''
COPY INTO {FQN}
FROM (
  SELECT
    CAST(order_id AS INT) AS order_id,
    CAST(customer_id AS INT) AS customer_id,
    CAST(order_date AS DATE) AS order_date,
    order_amount,
    order_status
  FROM '{SRC_DIR}')
FILEFORMAT = PARQUET
VALIDATE 10 ROWS
''').display()


order_id,customer_id,order_date,order_amount,order_status


## 9. Cleanup (optional)
Drop the demo tables and remove the sample files.


In [0]:
# Uncomment to clean up
# spark.sql(f'DROP TABLE IF EXISTS {FQN}')
# spark.sql(f'DROP TABLE IF EXISTS {FQN}_enriched')
# dbutils.fs.rm(SRC_DIR, recurse=True)
print('Done.')


---
### Quick reference

| Need | How |
|---|---|
| Load only new files | Just re-run the same `COPY INTO` (idempotent by default) |
| Reprocess all files | `COPY_OPTIONS ('force'='true')` |
| Add new columns to target | `COPY_OPTIONS ('mergeSchema'='true')` |
| Reconcile schema across source files | `FORMAT_OPTIONS ('mergeSchema'='true')` |
| Load a subset | `FILES = (...)` or `PATTERN = '...'` |
| Dry-run / check data | `VALIDATE [n ROWS]` |
| Transform during load | wrap source in `SELECT ... FROM '<path>'` |

**When to use COPY INTO vs Auto Loader:** Use `COPY INTO` for periodic batch ingestion of up to ~thousands of files. Use **Auto Loader** (`cloudFiles`) for continuous/streaming ingestion or when files number in the millions over time.

_Source: Databricks docs — COPY INTO (SQL language reference), last updated 2026-05-08._
